In [6]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np

In [7]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [18]:
mobileNet_model = keras.applications.MobileNet(
    input_shape=None,
    alpha=1.0,
    depth_multiplier=1,
    dropout=0.001,
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    pooling=None,
)

model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        mobileNet_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.4),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)

model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenet_1.00_224 (Functional) │ (None, 1, 1, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_674         │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_674 (Activation)     │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_675         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_675 (Activation)     │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 20)             │         5,140 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,893,204 (14.85 MB)

 Trainable params: 3,869,780 (14.76 MB)

 Non-trainable params: 23,424 (91.50 KB)

In [20]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 28s 47ms/step - accuracy: 0.1850 - loss: 2.7701 - val_accuracy: 0.3960 - val_loss: 1.9324
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.4712 - loss: 1.7246 - val_accuracy: 0.5610 - val_loss: 1.4103
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.5806 - loss: 1.3565 - val_accuracy: 0.6120 - val_loss: 1.2346
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.6463 - loss: 1.1273 - val_accuracy: 0.6270 - val_loss: 1.2164
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.7032 - loss: 0.9647 - val_accuracy: 0.6440 - val_loss: 1.1756


In [21]:
model.evaluate(x_test, testY)

63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.6492 - loss: 1.1187


[1.1044907569885254, 0.6464999914169312]